## Part 2: Prepare ELSET Data
#### 1. Acquire General Perturbation (gp) Data in form of Orbit Mean-Element Messages (OMMs) from [space-track.org](https://www.space-track.org/). Retrieve all OMMs for each object and then randomly select one Epoch for each object.
#### Last Updated: 2025-03-17

In [228]:
# Reauthenticate with Space-Track.org to avoid having to run first cell again

# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient
import pandas as pd

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.gp(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

Successfully authenticated with Space-Track.org!


In [151]:
# Read the NORAD IDs from the file
norad_ids = pd.read_csv("../Data/NoradIDs.csv")
print(f"Objects: {len(norad_ids)}")

Objects: 25798


##### Retrieve OMMs in chunks and adjust number of requested objects automatically when the number of returned objects gets too high. Save only Norad ID and Epoch to save RAM. 

In [ ]:
import time
import json
import os
from httpx import ReadTimeout
from spacetrack import SpaceTrackClient

# settings
chunk_size = 32
request_delay = 5
request_limit = 300
limit_delay = 1200
output_file = '../Data/AllEpochsLFS.csv'

# start variables
index = 22812
total_objects = len(norad_ids)
consecutive_timeouts = 0
consecutive_successful = 0
request_count = 1

# main loop
while index < total_objects:
    # Check if the API request limit has been reached
    if request_count >= request_limit:
        print(f"Reached {request_count} API requests. Waiting for {limit_delay} seconds.")
        time.sleep(limit_delay)
        request_count = 0

    # query start
    current_chunk = norad_ids[index:index + chunk_size] 
    api_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{','.join(current_chunk["NORAD_CAT_ID"].astype(str))}/format/json"
    print(f"\nRequest {request_count}: Querying {len(current_chunk)} objects from index {index} to {index+len(current_chunk)-1}...")
    print(f"API URL: {api_url}")


    # try fetching data for chunk
    try:
        request_count += 1
        time.sleep(request_delay)
        omm_data = st.gp_history(norad_cat_id=','.join(current_chunk["NORAD_CAT_ID"].astype(str)), format='json')
        omm_data_json = json.loads(omm_data)
        # check if data was returned
        if not omm_data_json:
            print("No data returned for this chunk. Skipping to the next chunk.")
            index += chunk_size
            consecutive_timeouts = 0
            consecutive_successful += 1
            continue
        omm_data_epochs = [{'NORAD_CAT_ID': entry['NORAD_CAT_ID'], 'EPOCH': entry['EPOCH']} for entry in omm_data_json] # extract only ID and Epoch to save RAM
    # catch exceptions
    except ReadTimeout:
        consecutive_timeouts += 1
        consecutive_successful = 0
        print(f"Timeout error ({consecutive_timeouts})")
        if consecutive_timeouts >= 2 and chunk_size > 1: # halve chunk size if api response is too large
            new_chunk_size = chunk_size // 2
            print(f"Two consecutive timeouts. Reducing chunk size from {chunk_size} to {new_chunk_size}.")
            chunk_size = new_chunk_size
            consecutive_timeouts = 0
        continue
    except Exception as e:
        print(f"Error: {e}")
        print("Skipping to the next chunk.")
        index += chunk_size
        consecutive_timeouts = 0
        consecutive_successful = 0
        continue

    consecutive_timeouts = 0
    consecutive_successful += 1

    df_chunk = pd.DataFrame(omm_data_epochs)

    # log results
    num_ids_with_omm = df_chunk['NORAD_CAT_ID'].nunique()
    total_omms = len(df_chunk)
    print(f"Chunk {index // chunk_size}: Requested {len(current_chunk)} objects, {num_ids_with_omm} returned at least one OMM, {total_omms} OMMs in total retrieved.")

    df_chunk.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
    index += chunk_size

    # double chunk size every 5 iterations (after 15 iterations (of 500) since 1000 would be a too long url) to see if bigger chunks are possible
    if consecutive_successful % 5 == 0:
        chunk_size = chunk_size * 2
        print(f"Five successful iterations passed. Doubling chunk size from {chunk_size // 2} to {chunk_size}.")



Request 1: Querying 4 objects from index 22812 to 22815...
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/34447,34493,34496,34522/format/json
Timeout error (1)

Request 2: Querying 4 objects from index 22812 to 22815...
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/34447,34493,34496,34522/format/json
Timeout error (2)
Two consecutive timeouts. Reducing chunk size from 4 to 2.

Request 3: Querying 2 objects from index 22812 to 22813...
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/34447,34493/format/json
Chunk 11406: Requested 2 objects, 2 returned at least one OMM, 15267 OMMs in total retrieved.

Request 4: Querying 2 objects from index 22814 to 22815...
API URL: https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/34496,34522/format/json
Chunk 11407: Requested 2 objects, 2 returned at least one OMM, 12154 OMMs in total retrieved.

Requ

In [181]:
# check how many epochs were retrieved in total
retrieved_epochs = pd.read_csv(output_file)
print(f"Total OMMs retrieved: {len(retrieved_epochs)}")

Total OMMs retrieved: 28545387


##### Randomly select one Epoch per object.

In [187]:
# Load the data
AllEpochs_df = pd.read_csv('../Data/AllEpochsLFS.csv')

# Group by NORAD_CAT_ID and randomly select one epoch for each group
RandomEpochs_df = AllEpochs_df.groupby('NORAD_CAT_ID', group_keys=False).apply(lambda x: x.sample(1, random_state=1)).reset_index(drop=True)

# Save the result to a new CSV file
RandomEpochs_df.to_csv('../Data/RandomEpochs.csv', index=False)

print(f"Randomly selected one epoch for each of the {len(RandomEpochs_df)} unique NORAD IDs. {len(norad_ids) - len(RandomEpochs_df)} are missing OMM data.")

Randomly selected one epoch for each of the 24474 unique NORAD IDs. 1324 are missing OMM data.


/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_24366/932978534.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  RandomEpochs_df = AllEpochs_df.groupby('NORAD_CAT_ID', group_keys=False).apply(lambda x: x.sample(1, random_state=1)).reset_index(drop=True)


#### 2. Query the full OMM for each object for the randomly selected Epoch.

In [231]:
def re_login():
    # read Space-Track.org credentials from external file (username in line 1, password in line 2)
    with open('../SpaceTrackCredentials.txt', 'r') as file:
        credentials = file.read().splitlines()

    username = credentials[0]
    password = credentials[1]

    # create a SpaceTrackClient instance
    st = SpaceTrackClient(username, password) 

    try:
        # Try fetching a single SATCAT entry to verify authentication
        test_data = st.gp(limit=1, format='json')
        print("Successfully re-logged in to Space-Track.org!")
    except Exception as e:
        print(f"Failed to authenticate with Space-Track.org: {e}")

In [ ]:
import time
import json
from datetime import datetime, timedelta

# Load the data
RandomEpochs_df = pd.read_csv('../Data/RandomEpochs.csv')

# DataFrame to store full OMM data if first execution
OMMs_list = []

# define start index
start_index = 0

# Iterate over the DataFrame and query full OMM for each NORAD ID and Epoch
for index, row in RandomEpochs_df.iloc[start_index:].iterrows():
    norad_id = row['NORAD_CAT_ID']
    epoch = row['EPOCH']
    
    # Extract the date part from the epoch
    start_date = epoch.split("T")[0]
    end_date = (datetime.strptime(start_date, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
    api_url = f"https://www.space-track.org/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{norad_id}/EPOCH/>{start_date},<{end_date}/limit/1/format/json"
    
    retry = True
    retry_count = 0

    while retry and retry_count < 3:
        retry = False
        retry_count += 1
         # try fetching data
        try:
            response = st.gp_history(norad_cat_id=norad_id, epoch=f">{start_date},<{end_date}", limit=1, format='json')
            # Retry once after 60 seconds if rate-limited
            if "You've violated your query rate limit" in response:
                print(f"({index}) *** Rate limit hit for NORAD ID {norad_id}. Retrying in 60 seconds... ***")
                time.sleep(60)
                response = st.gp_history(norad_cat_id=norad_id, epoch=f">{start_date},<{end_date}", limit=1, format='json')
                # Skip if still rate-limited
                if "You've violated your query rate limit" in response:
                    print(f"({index}) *** Still rate-limited after retry. Skipping NORAD ID {norad_id} ***")
                    continue
            OMMs_list.extend(json.loads(response))
            print(f"({index}) Successfully retrieved OMM data for NORAD ID {norad_id}")
        # catch exceptions
        except Exception as e:
            if "401" in str(e):
                print(f"({index}) *** Error: {e} \nRe-authenticating...")
                re_login()
                retry = True
            else:
                print(f"({index}) *** Error: {e} for NORAD ID {norad_id} *** \nAPI URL: {api_url} \nWaiting for 60 seconds before retrying...")
                time.sleep(60)
                retry = True
        
    if retry_count >= 3:
        print(f"({index}) *** Skipping NORAD ID {norad_id} after 3 retries ***")
    
    # Save data to CSV every 10 iterations
    if (index + 1) % 100 == 0:
        OMMs_df = pd.DataFrame(OMMs_list)
        OMMs_df.to_csv('../Data/OMMs.csv', index=False)
        print(f"Saved OMM data to csv. Total OMMs retrieved: {len(OMMs_df)}")

# Save the final data to a CSV file
OMMs_df = pd.DataFrame(OMMs_list)
OMMs_df.to_csv('../Data/OMMs_final.csv', index=False)

print(f"\nRetrieved full OMM data for {len(OMMs_df)} NORAD IDs. {len(RandomEpochs_df) - len(OMMs_df)} objects are missing OMM data.")

(0) Successfully retrieved OMM data for NORAD ID 23
(1) Successfully retrieved OMM data for NORAD ID 28
(2) Successfully retrieved OMM data for NORAD ID 30
(3) Successfully retrieved OMM data for NORAD ID 33
(4) Successfully retrieved OMM data for NORAD ID 35
(5) Successfully retrieved OMM data for NORAD ID 37
(6) Successfully retrieved OMM data for NORAD ID 38
(7) Successfully retrieved OMM data for NORAD ID 39
(8) Successfully retrieved OMM data for NORAD ID 40
(9) Successfully retrieved OMM data for NORAD ID 41
(10) Successfully retrieved OMM data for NORAD ID 42
(11) Successfully retrieved OMM data for NORAD ID 44
(12) Successfully retrieved OMM data for NORAD ID 56
(13) Successfully retrieved OMM data for NORAD ID 62
(14) Successfully retrieved OMM data for NORAD ID 64
(15) Successfully retrieved OMM data for NORAD ID 66
(16) Successfully retrieved OMM data for NORAD ID 69
(17) Successfully retrieved OMM data for NORAD ID 72
(18) Successfully retrieved OMM data for NORAD ID 73
(19

#### 3. Inspect data to see which columns are unneccecary 

In [7]:
# inspect certain columns to see if they can be dropped
print(OMMs_df['ORIGINATOR'].unique())
print(OMMs_df['CENTER_NAME'].unique())
print(OMMs_df['REF_FRAME'].unique())
print(OMMs_df['TIME_SYSTEM'].unique())
print(OMMs_df['MEAN_ELEMENT_THEORY'].unique())
print(OMMs_df['EPHEMERIS_TYPE'].unique())
print(OMMs_df['CLASSIFICATION_TYPE'].unique())
print(OMMs_df['ELEMENT_SET_NO'].unique())

['18 SPCS']
['EARTH']
['TEME']
['UTC']
['SGP4']
['0']
['U']
['999']


In [8]:
# drop not needed columns
OMMs_df.drop(columns=['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'COUNTRY_CODE'], inplace=True)
OMMs_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE
0,THOR ABLESTAR R/B,1960-003A,1961-08-16T14:24:53.774496,16.29358349,0.00000000,51.2500,356.4892,331.9473,28.0526,30,7590,0.00000000000000,0.02125614,0.0000000000000,6572.405,88.378,194.270,194.270,ROCKET BODY,None,None,None,1961-08-18
1,THOR ABLESTAR DEB,1960-003C,1960-10-06T07:59:06.496512,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17
2,SL-3 R/B,1960-005B,1960-07-16T04:00:11.150208,16.27523673,0.00000000,64.8899,44.6976,36.3819,323.6180,35,990,0.00000000000000,0.03271057,0.0000000000000,6577.344,88.477,199.209,199.209,ROCKET BODY,None,None,None,1960-07-17
3,SPUTNIK 4 DEB,1960-005D,1961-03-15T15:48:42.802271,15.49800338,0.01984030,64.8899,343.2411,333.4201,27.0840,37,4690,0.00000000000000,0.00039007,0.0000000000000,6795.450,92.915,552.139,282.492,DEBRIS,None,None,None,1961-06-30
4,SPUTNIK 4 DEB,1960-005E,1960-08-19T16:34:33.722976,15.88909013,0.00342130,64.8899,302.1218,49.4823,310.6691,38,1500,0.00000000000000,0.00610805,0.0000000000000,6683.481,90.628,328.212,282.480,DEBRIS,None,None,None,1960-08-20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24678,FALCON 9 R/B,2025-022Y,2025-02-19T02:13:45.786720,16.47531719,0.00310630,53.1436,118.6486,5.7660,67.2814,62878,274,0.00011008000000,0.10554311,0.0000130660000,6523.980,87.403,166.111,125.580,ROCKET BODY,LARGE,2025-02-01,AFWTR,2025-02-19
24679,VOLGA R/B,2025-026D,2025-02-05T11:52:04.781856,14.93887108,0.00100970,81.9945,220.5351,276.0393,83.9657,62905,3,0.00656530000000,0.00070488,0.0000000000000,6963.966,96.393,592.863,578.800,ROCKET BODY,None,None,None,2025-02-05
24680,ELECTRON R/B,2025-028F,2025-02-20T21:10:11.146080,16.39810484,0.00545400,97.9272,109.5562,197.5517,275.2945,62933,187,0.00027058000000,0.13714693,0.0000021991000,6544.444,87.815,202.002,130.615,ROCKET BODY,MEDIUM,2025-02-08,RLLC,2025-02-21
24681,ELECTRON R/B,2025-033C,2025-02-26T00:20:35.861280,16.43070160,0.00529660,58.9964,75.2812,270.2695,186.1265,63028,112,0.00030531000000,0.25107025,0.0000108760000,6535.785,87.641,192.268,123.033,ROCKET BODY,MEDIUM,2025-02-18,RLLC,2025-02-26


In [9]:
# save to csv
OMMs_df.to_csv("../Data/OMMs.csv", index=False)

#### 3. Check the differences between Decay Date and Epoch (timestamp of OMM)

In [10]:
# convert to date and then again to datetime, needed because otherwise the date conversion does not work :/
OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH']).dt.date
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE']).dt.date

OMMs_df['EPOCH'] = pd.to_datetime(OMMs_df['EPOCH'])
OMMs_df['DECAY_DATE'] = pd.to_datetime(OMMs_df['DECAY_DATE'])

In [11]:
# check difference between Epoch and Decay Date
OMMs_df['Epoch_Decay_Difference'] = (OMMs_df['DECAY_DATE']-OMMs_df['EPOCH'])
OMMs_df['Epoch_Decay_Difference'].describe()

count                          24683
mean      19 days 05:21:06.118381072
std      300 days 09:36:12.104763172
min            -10727 days +00:00:00
25%                  0 days 00:00:00
50%                  1 days 00:00:00
75%                  5 days 00:00:00
max               8300 days 00:00:00
Name: Epoch_Decay_Difference, dtype: object

In [12]:
# show objects where Epoch_Decay_Difference is negative
OMMs_wrong_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] < pd.Timedelta(0)]
OMMs_wrong_df

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,SEMIMAJOR_AXIS,PERIOD,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,LAUNCH_DATE,SITE,DECAY_DATE,Epoch_Decay_Difference
1,THOR ABLESTAR DEB,1960-003C,1960-10-06,15.38343354,0.02427570,51.2879,136.0668,196.7800,163.6100,33,2700,0.00000000000000,0.00001932,0.0000000000000,6829.149,93.607,616.796,285.231,DEBRIS,None,None,None,1960-07-17,-81 days
2872,DELTA 1 DEB,1972-058C,2004-11-24,15.86540389,0.00128020,98.3597,332.5522,114.4864,245.7741,7832,57786,0.00141360000000,0.00371465,0.0000000000000,6690.132,90.763,320.561,303.432,DEBRIS,LARGE,1972-07-23,AFWTR,1977-02-12,-10147 days
3087,COSMOS 777 DEB,1975-102F,1976-05-02,15.50607889,0.00266690,65.0265,340.8773,6.7572,353.3878,8625,2876,0.00354340000000,0.00212521,0.0000000000000,6793.091,92.866,433.072,396.839,DEBRIS,None,None,None,1976-03-15,-48 days
3100,COSMOS 777 DEB,1975-102AE,1976-05-02,15.50641491,0.00104770,65.0344,341.0843,8.9752,351.1527,8650,2877,0.00000000000000,0.00246035,0.0000000000000,6792.993,92.864,421.975,407.741,DEBRIS,None,None,None,1976-03-27,-36 days
3142,COSMOS 777 DEB,1975-102BF,1976-05-02,15.40156937,0.00557380,64.8281,339.9362,23.9313,336.4347,8728,2872,0.00328330000000,0.00142121,0.0000000000000,6823.787,93.496,483.686,407.617,DEBRIS,None,None,None,1976-03-29,-34 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23254,BION 11 DEB,1996-073G,1997-02-03,16.25788645,0.00087860,62.7868,87.1680,23.0743,337.0808,24710,439,0.00004231000000,0.00130341,0.0000086059000,6582.022,88.572,209.670,198.105,DEBRIS,SMALL,1996-12-24,PKMTR,1997-01-10,-24 days
23897,CZ-2F DEB,2024-078D,2024-04-28,16.31592083,0.00635030,41.5973,28.4800,152.0982,294.6586,59594,37,0.00245460000000,0.38721847,0.0000072850000,6566.402,88.257,229.965,146.568,DEBRIS,None,2024-04-25,JSC,2024-04-27,-1 days
24120,COSMOS 2251 DEB,1993-036AZQ,2024-05-11,14.67237796,0.00346710,73.9910,229.2548,89.3171,271.1971,36387,57450,0.02934800000000,0.00171058,0.0000000000000,7048.037,98.144,694.338,645.466,DEBRIS,SMALL,1993-06-16,PKMTR,2024-05-07,-4 days
24312,COSMOS 1408 DEB,1982-092CCH,2023-07-06,15.17775450,0.01267330,82.5778,267.2573,149.0686,211.8095,57287,1,0.00109990000000,0.00027530,0.0000000000000,6890.702,94.876,599.895,425.239,DEBRIS,None,None,None,2023-06-12,-24 days


In [13]:
# save IDs of these objects to csv to later also delete from SATCAT data
wrong_ids = OMMs_wrong_df['NORAD_CAT_ID']
wrong_ids.to_csv("../Data/WrongIDs.csv", index=False)

In [14]:
# drop objects where OMM recorded after object was supposed to have decayed
OMMs_df = OMMs_df[OMMs_df['Epoch_Decay_Difference'] >= pd.Timedelta(0)]
print(f"Remaining objects: {len(OMMs_df)}")



Remaining objects: 24257


In [15]:
# Desribe in days
OMMs_df['Epoch_Decay_Difference'] = OMMs_df['Epoch_Decay_Difference'].dt.days
OMMs_df['Epoch_Decay_Difference'].describe()

/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_4716/1006036595.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OMMs_df['Epoch_Decay_Difference'] = OMMs_df['Epoch_Decay_Difference'].dt.days


count    24257.000000
mean        34.496805
std        222.906075
min          0.000000
25%          0.000000
50%          1.000000
75%          5.000000
max       8300.000000
Name: Epoch_Decay_Difference, dtype: float64

In [16]:
# remove the difference and epoch column 
OMMs_df.drop(columns=['EPOCH', 'Epoch_Decay_Difference'], inplace=True)

/var/folders/xm/n65_n9sd78z4_zdw8r87klf00000gn/T/ipykernel_4716/1425059375.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  OMMs_df.drop(columns=['EPOCH', 'Epoch_Decay_Difference'], inplace=True)


In [17]:
# Save OMMs to csv
OMMs_df.to_csv('../Data/OMMs.csv', index=False)